# 13 - Overlay: theme conviction vs the theme's ETF price

Theme **conviction_z** (`daily_theme_conviction`) vs the price of the ETF the
theme is anchored to (`THEME_ETFS`). Clipped to the `update_data.py` window.
Edit `THEME` and re-run.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

# find project root whether run from notebooks/ or the project root
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# THE WINDOW comes from update_data.py - the SAME two dates that control live vs
# backtest for the whole pipeline. END_DATE = '' means live (up to the newest
# data); set END_DATE to a date in update_data.py to freeze a past regime.
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    """Keep only the dates inside the window (s indexed by date)."""
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    """Keep only the rows whose date column is inside the window."""
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError(
            'prices.parquet not found - run  python pull_bloomberg_prices.py  '
            'on the work laptop first (needs the Bloomberg Terminal).')
    px = pd.read_parquet(PRICES_PATH)
    px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    one = prices[prices['symbol'] == symbol].sort_values('date')
    if one.empty:
        print('WARNING: no price rows for', symbol, '- is it in the pull universe?')
    return clip_series(one.set_index('date')['px_last'])


In [ ]:
from src.themes import THEME_ETFS
print('themes available:', ', '.join(sorted(THEME_ETFS)))

In [ ]:
# ============ EDIT THIS ============
THEME = 'ai_megacap'    # which theme (see the list printed above)
# ===================================

In [ ]:
conv = pd.read_parquet(os.path.join(P, 'daily_theme_conviction.parquet'))
conv['date'] = pd.to_datetime(conv['date'])
c = conv[conv['theme'] == THEME].sort_values('date').set_index('date')['conviction_z']
c = clip_series(c)                       # focus the window from update_data.py

etf = THEME_ETFS.get(THEME)
if etf is None:
    raise KeyError(f'{THEME} is not in THEME_ETFS - pick one from the list above.')

prices = load_prices()
px = price_series(prices, etf)

fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.axhline(0, color='black', linewidth=0.6)
ax1.plot(c.index, c.values, color='tab:purple', linewidth=1.6,
         label=f'{THEME} conviction_z')
ax1.set_ylabel('conviction z-score', color='tab:purple')
ax1.tick_params(axis='y', labelcolor='tab:purple')

ax2 = ax1.twinx()
ax2.plot(px.index, px.values, color='tab:red', linewidth=1.6,
         label=f'{etf} price (PX_LAST)')
ax2.set_ylabel('price (USD)', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

ax1.set_title(f'{THEME}: conviction vs {etf} price')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()